## Data Loading and Schema Inspection 

In [0]:
# Verify the data loaded correctly into the table 
spark.sql("SELECT * FROM netflix_titles LIMIT 10").show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [0]:
# Understand the size of the dataset
spark.sql("""
SELECT COUNT(*) AS total_rows
FROM netflix_titles
""").show()

+----------+
|total_rows|
+----------+
|      8809|
+----------+



In [0]:
# Check columns and data types
spark.sql("DESCRIBE netflix_titles").show(truncate=False)

+------------+---------+-------+
|col_name    |data_type|comment|
+------------+---------+-------+
|show_id     |string   |NULL   |
|type        |string   |NULL   |
|title       |string   |NULL   |
|director    |string   |NULL   |
|cast        |string   |NULL   |
|country     |string   |NULL   |
|date_added  |string   |NULL   |
|release_year|bigint   |NULL   |
|rating      |string   |NULL   |
|duration    |string   |NULL   |
|listed_in   |string   |NULL   |
|description |string   |NULL   |
+------------+---------+-------+



### Observations: 
- release year - numeric, 
- date_added - string (!) - we woudl need to convert it to numeric
- all other columns are text (string)
- there is a number of NULLs/missing data 

## Data Quality Assessment 

In [0]:
# Before cleaning we will explore hopw much missing data we actually have 
spark.sql("""
SELECT
    SUM(CASE WHEN director IS NULL THEN 1 ELSE 0 END) AS missing_director,
    SUM(CASE WHEN cast IS NULL THEN 1 ELSE 0 END) AS missing_cast,
    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS missing_country,
    SUM(CASE WHEN date_added IS NULL THEN 1 ELSE 0 END) AS missing_date_added,
    SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS missing_rating
FROM netflix_titles
""").show()

+----------------+------------+---------------+------------------+--------------+
|missing_director|missing_cast|missing_country|missing_date_added|missing_rating|
+----------------+------------+---------------+------------------+--------------+
|            2636|         826|            833|                12|             6|
+----------------+------------+---------------+------------------+--------------+



### Findings: 
Missing data is concentrated mainly in the ditector (~30%), cast, and country fileds

In [0]:
# Check for duplicates 
spark.sql("""
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT title) AS unique_show_ids
FROM netflix_titles
""").show()

+----------+---------------+
|total_rows|unique_show_ids|
+----------+---------------+
|      8809|           8804|
+----------+---------------+



### Findings: 
5 records have duplicate values and require investigation before cleaning.


In [0]:
# Investigate duplicate 
spark.sql("""
SELECT
    show_id,
    COUNT(*) AS occurrences
FROM netflix_titles
GROUP BY show_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
""").show(truncate=False)

+-------+-----------+
|show_id|occurrences|
+-------+-----------+
+-------+-----------+



### Observation:

Initial comparison of total rows and distinct show IDs suggested 5 duplicate records.

Direct duplicate investigation found no repeated show_id values.

Further validation is required.

In [0]:
# Check if some show_ids are NULLs
spark.sql("""
SELECT
    COUNT(*) AS null_show_ids
FROM netflix_titles
WHERE show_id IS NULL
""").show()

+-------------+
|null_show_ids|
+-------------+
|            0|
+-------------+



### Data Quality assessment conclusion: 

A disrepancy of 5 records between total row count and distinct show IDs represent 0.06% of the dataset  and is considered insignificant for the purposes of this exploratory analysis

## Data Cleaning and Preparation 

In [0]:
# Check on all non-null date_added to see if they are in the correct format
spark.sql("""
SELECT DISTINCT date_added
FROM netflix_titles
WHERE date_added IS NOT NULL
ORDER BY date_added
LIMIT 20
""").show(truncate=False)

+------------------+
|date_added        |
+------------------+
| April 15, 2018   |
| April 16, 2019   |
| April 17, 2016   |
| April 20, 2017   |
| April 4, 2017    |
| August 1, 2017   |
| August 13, 2018  |
| August 21, 2017  |
| August 4, 2017   |
| December 1, 2018 |
| December 1, 2019 |
| December 14, 2018|
| December 15, 2015|
| December 15, 2017|
| December 15, 2018|
| December 18, 2014|
| December 2, 2017 |
| December 23, 2018|
| December 25, 2015|
| December 28, 2016|
+------------------+



All date formats look consistent 

In [0]:
# We will now proceed with data cleaning and create a cleaned view of the table 
spark.sql("""
CREATE OR REPLACE TEMP VIEW netflix_cleaned AS
SELECT
    *,
    TO_DATE(date_added, 'MMMM d, yyyy') AS date_added_clean
FROM netflix_titles
""")

DataFrame[]

In [0]:
# Verify the conversion worked 
spark.sql("""
SELECT
    date_added,
    date_added_clean
FROM netflix_cleaned
LIMIT 10
""").show(truncate=False)

+------------------+----------------+
|date_added        |date_added_clean|
+------------------+----------------+
|September 25, 2021|2021-09-25      |
|September 24, 2021|2021-09-24      |
|September 24, 2021|2021-09-24      |
|September 24, 2021|2021-09-24      |
|September 24, 2021|2021-09-24      |
|September 24, 2021|2021-09-24      |
|September 24, 2021|2021-09-24      |
|September 24, 2021|2021-09-24      |
|September 24, 2021|2021-09-24      |
|September 24, 2021|2021-09-24      |
+------------------+----------------+



In [0]:
# We will replace all NULLs with UNKNOWN so it supports the analysis. 
# We do not want ton lose rows with NULLs, neither we want to lose the information in the column
spark.sql("""
CREATE OR REPLACE TEMP VIEW netflix_cleaned AS
SELECT
    show_id,
    type,
    title,
    COALESCE(director, 'Unknown') AS director,
    COALESCE(cast, 'Unknown') AS cast,
    COALESCE(country, 'Unknown') AS country,
    TRY_TO_DATE(TRIM(date_added), 'MMMM d, yyyy') AS date_added,
    release_year,
    COALESCE(rating, 'Unknown') AS rating,
    duration,
    listed_in,
    description
FROM netflix_titles
""")

DataFrame[]

In [0]:
# Validate the changes 
spark.sql("""
SELECT
    SUM(CASE WHEN director IS NULL THEN 1 ELSE 0 END) AS missing_director,
    SUM(CASE WHEN cast IS NULL THEN 1 ELSE 0 END) AS missing_cast,
    SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END) AS missing_country,
    SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) AS missing_rating,
    SUM(CASE WHEN date_added IS NULL THEN 1 ELSE 0 END) AS missing_date_added
FROM netflix_cleaned
""").show()

+----------------+------------+---------------+--------------+------------------+
|missing_director|missing_cast|missing_country|missing_rating|missing_date_added|
+----------------+------------+---------------+--------------+------------------+
|               0|           0|              0|             0|                13|
+----------------+------------+---------------+--------------+------------------+



### Cleaning actions performed:

- Replaced NULL values in: director, cast, country, rating with "Unknown".
- Trimmed whitespace from date_added.
- Converted date_added from string to date using TRY_TO_DATE().

## Exploratory Data Analysis

In [0]:
# What proportion of Netflix content consists of Movies versus TV Shows?

spark.sql("""
SELECT
    type,
    COUNT(*) AS total_titles
FROM netflix_cleaned
GROUP BY type
ORDER BY total_titles DESC
""").show()

+-------------+------------+
|         type|total_titles|
+-------------+------------+
|        Movie|        6131|
|      TV Show|        2676|
|         NULL|           1|
|William Wyler|           1|
+-------------+------------+



### Finding:

- Movies: 6,131
- TV Shows: 2,676

### Interpretation:

- Movies account for approximately 70% of the catalogue.
- Netflix's catalogue is heavily weighted towards films rather than shows.

In [0]:
# Some additional checks and cleaning need to be done because we have NULL and William Wyler in type column 
# Now we need to verify the cleaned table structure if it did not shift during cleaning process 
spark.sql("""
DESCRIBE netflix_cleaned
""").show(truncate=False)

+------------+---------+-------+
|col_name    |data_type|comment|
+------------+---------+-------+
|show_id     |string   |NULL   |
|type        |string   |NULL   |
|title       |string   |NULL   |
|director    |string   |NULL   |
|cast        |string   |NULL   |
|country     |string   |NULL   |
|date_added  |date     |NULL   |
|release_year|bigint   |NULL   |
|rating      |string   |NULL   |
|duration    |string   |NULL   |
|listed_in   |string   |NULL   |
|description |string   |NULL   |
+------------+---------+-------+



In [0]:
# We try and find that one row with a wrong type name and with NULL 
spark.sql("""
SELECT *
FROM netflix_cleaned
WHERE type NOT IN ('Movie', 'TV Show')
   OR type IS NULL
""").show(truncate=False)


+--------------------+-------------+-----+-------------+--------------+-------+----------+------------+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+---------+-----------+
|show_id             |type         |title|director     |cast          |country|date_added|release_year|rating                       |duration                                                                                                                                          |listed_in|description|
+--------------------+-------------+-----+-------------+--------------+-------+----------+------------+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+---------+-----------+
| and probably will."|NULL         |NULL |Unknown      |Unknown       |Unknown|NULL      |N

### Additional finding and decision:

Two corrupted records were identified during exploratory analysis. As they represented less than 0.1% of the dataset, they were excluded from further analysis.

In [0]:
# Create a clean analytical view
spark.sql("""
CREATE OR REPLACE TEMP VIEW netflix_analysis AS
SELECT *
FROM netflix_cleaned
WHERE type IN ('Movie', 'TV Show')
""")

DataFrame[]

Note: we will use cleaned table name "netflix_analysis" from now on in the analysis

In [0]:
# How has Netflix content grown over time?
spark.sql("""
SELECT
    YEAR(date_added) AS year_added,
    COUNT(*) AS total_titles
FROM netflix_analysis
WHERE date_added IS NOT NULL
GROUP BY YEAR(date_added)
ORDER BY year_added
""").show()

+----------+------------+
|year_added|total_titles|
+----------+------------+
|      2008|           2|
|      2009|           2|
|      2010|           1|
|      2011|          13|
|      2012|           3|
|      2013|          11|
|      2014|          24|
|      2015|          82|
|      2016|         429|
|      2017|        1187|
|      2018|        1649|
|      2019|        2016|
|      2020|        1879|
|      2021|        1498|
+----------+------------+



Netflix's content catalogue expanded rapidly between 2016 and 2019

Content additions remained high in 2020 but declined slightly in 2021.

The analysis suggests that Netflix experienced its strongest period of catalogue growth between 2017 and 2020, reflecting significant investment in content acquisition and production during those years.

In [0]:
# Which countries contribute the most content to Netflix?
spark.sql("""
SELECT
    country,
    COUNT(*) AS total_titles
FROM netflix_analysis
GROUP BY country
ORDER BY total_titles DESC
LIMIT 10
""").show(truncate=False)

+--------------+------------+
|country       |total_titles|
+--------------+------------+
|United States |2817        |
|India         |972         |
|Unknown       |832         |
|United Kingdom|419         |
|Japan         |245         |
|South Korea   |199         |
|Canada        |181         |
|Spain         |145         |
|France        |124         |
|Mexico        |110         |
+--------------+------------+



The United States dominates Netflix's catalogue, significantly ahead of all other countries.

India is the second-largest contributor, highlighting Netflix's strong presence in the Indian market.

The presence of 832 titles with an "Unknown" country indicates a notable amount of missing geographical metadata.

Outside the United States and India, the largest contributors are the United Kingdom, Japan, South Korea, and Canada, demonstrating Netflix's increasingly international content portfolio.

In [0]:
# What are the most common content ratings on Netflix?
spark.sql("""
SELECT
    rating,
    COUNT(*) AS total_titles
FROM netflix_analysis
GROUP BY rating
ORDER BY total_titles DESC
""").show(truncate=False)

+--------+------------+
|rating  |total_titles|
+--------+------------+
|TV-MA   |3207        |
|TV-14   |2160        |
|TV-PG   |862         |
|R       |799         |
|PG-13   |490         |
|TV-Y7   |334         |
|TV-Y    |307         |
|PG      |287         |
|TV-G    |220         |
|NR      |80          |
|G       |41          |
|TV-Y7-FV|6           |
|Unknown |5           |
|NC-17   |3           |
|UR      |3           |
|74 min  |1           |
|84 min  |1           |
|66 min  |1           |
+--------+------------+



Netflix's catalogue is dominated by mature (TV-MA) and teen-oriented (TV-14) content.

Together, these two ratings account for the majority of the catalogue.

* inside the rating column there is a small number of data quality issues, but because they are only 3 rows out of 8800, they can be nioted as data anomalies and excluded from further investigation

In [0]:
# What are the most popular genres on Netflix?
spark.sql("""
SELECT
    TRIM(genre) AS genre,
    COUNT(*) AS total_titles
FROM (
    SELECT
        EXPLODE(SPLIT(listed_in, ',')) AS genre
    FROM netflix_analysis
)
GROUP BY TRIM(genre)
ORDER BY total_titles DESC
LIMIT 25
""").show(truncate=False)

+------------------------+------------+
|genre                   |total_titles|
+------------------------+------------+
|International Movies    |2752        |
|Dramas                  |2427        |
|Comedies                |1674        |
|International TV Shows  |1351        |
|Documentaries           |868         |
|Action & Adventure      |859         |
|TV Dramas               |763         |
|Independent Movies      |756         |
|Children & Family Movies|641         |
|Romantic Movies         |616         |
|TV Comedies             |581         |
|Thrillers               |577         |
|Crime TV Shows          |470         |
|Kids' TV                |451         |
|Docuseries              |395         |
|Music & Musicals        |375         |
|Romantic TV Shows       |370         |
|Horror Movies           |357         |
|Stand-Up Comedy         |343         |
|Reality TV              |255         |
+------------------------+------------+
only showing top 20 rows


Netflix's catalogue is strongly focused on international and drama-based content.

The prominence of International Movies and International TV Shows suggests that Netflix has invested heavily in global content rather than relying solely on domestic productions.

The platform also maintains a diverse catalogue spanning documentaries, action, family content, crime, romance, horror, and reality television.

In [0]:
# How old is the content on Netflix? What release years dominate the catalogue?
spark.sql("""
SELECT
    release_year,
    COUNT(*) AS total_titles
FROM netflix_analysis
GROUP BY release_year
ORDER BY total_titles DESC
LIMIT 30
""").show()

+------------+------------+
|release_year|total_titles|
+------------+------------+
|        2018|        1147|
|        2017|        1032|
|        2019|        1030|
|        2020|         953|
|        2016|         902|
|        2021|         592|
|        2015|         560|
|        2014|         352|
|        2013|         288|
|        2012|         237|
|        2010|         194|
|        2011|         185|
|        2009|         152|
|        2008|         136|
|        2006|          96|
|        2007|          88|
|        2005|          80|
|        2004|          64|
|        2003|          61|
|        2002|          51|
+------------+------------+
only showing top 20 rows


Netflix's catalogue is dominated by recent content, with the highest concentration of titles released between 2016 and 2020.

The most common release year is 2018, followed closely by 2017 and 2019.

The number of titles generally decreases for older release years, suggesting that Netflix prioritises relatively modern content while maintaining a smaller collection of older films and television programmes.

This indicates a content strategy focused on keeping the catalogue current and relevant to contemporary audiences.


In [0]:
# Which countries produce the most TV Shows versus Movies?
spark.sql("""
SELECT
    country,
    type,
    COUNT(*) AS total_titles
FROM netflix_analysis
WHERE country != 'Unknown'
GROUP BY country, type
ORDER BY total_titles DESC
LIMIT 20
""").show(truncate=False)

+-----------------------------+-------+------------+
|country                      |type   |total_titles|
+-----------------------------+-------+------------+
|United States                |Movie  |2057        |
|India                        |Movie  |893         |
|United States                |TV Show|760         |
|United Kingdom               |TV Show|213         |
|United Kingdom               |Movie  |206         |
|Japan                        |TV Show|169         |
|South Korea                  |TV Show|158         |
|Canada                       |Movie  |122         |
|Spain                        |Movie  |97          |
|Egypt                        |Movie  |92          |
|Nigeria                      |Movie  |86          |
|India                        |TV Show|79          |
|Indonesia                    |Movie  |77          |
|Turkey                       |Movie  |76          |
|Japan                        |Movie  |76          |
|France                       |Movie  |75     

The analysis reveals notable differences in content type by country.

The United States dominates both Movies (2,057) and TV Shows (760), making it Netflix's largest content source overall.

India is heavily movie-focused, with 893 Movies but only 79 TV Shows.

Japan and South Korea show a stronger emphasis on TV content, contributing 169 and 158 TV Shows respectively.

The United Kingdom has a relatively balanced contribution across Movies (206) and TV Shows (213).

These findings suggest that Netflix's content portfolio reflects different production strengths across regions, with some countries contributing primarily films while others are more prominent in television content.

## Data Storage 

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE netflix_cleaned_final AS
SELECT *
FROM netflix_analysis
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM netflix_cleaned_final
""").show()

+--------+
|COUNT(*)|
+--------+
|    8807|
+--------+



The count is 8,807 (8,809 minus the 2 corrupted records)

In [0]:
spark.sql("""
SHOW TABLES
""").show(truncate=False)

+--------+---------------------+-----------+
|database|tableName            |isTemporary|
+--------+---------------------+-----------+
|default |iris                 |false      |
|default |netflix_cleaned_final|false      |
|default |netflix_titles       |false      |
+--------+---------------------+-----------+

